# Brain signal reconstruction
Waveforms and masking curves based on `draw_pic/fig4.ipynb` and `signal_recon.ipynb`.
Masks are explicit boolean arrays: **zero-valued signal entries are not treated as missing**.
Temporal blocks and channels are sampled without replacement, with a fixed seed.
Missing-entry and observed-entry errors are reported separately.
A clean-trained checkpoint measures robustness; only a verified mask-trained checkpoint
supports a mask-trained reconstruction claim. Select the example index before viewing results.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or the release root.
ROOT = Path.cwd() if (Path.cwd() / "brainae").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figure_utils import *
style()
OUTPUT = ROOT / "figures"


In [ ]:
MODALITY='eeg'
DATA=ROOT/'data/prepared/eeg/sub-01'
CHECKPOINT=ROOT/'outputs/eeg/sub-01/best.pt'
PROTOTYPES=ROOT/'data/features/eeg_center.npy'
DEVICE='cpu'
CHECKPOINT_TRAINING='clean-input training'  # State the actual training protocol.
PROTOTYPE_METRIC='original-dot'
SFREQ={'eeg':250,'meg':200,'mua':1000}[MODALITY]
T0=.005 if MODALITY=='meg' else 0.
RATIOS=[.25,.5,.75]
KINDS=['random','temporal','spatial','forecasting']
EXAMPLE_IMAGE=0
EXAMPLE_CHANNELS=[0]  # Use verified O1/Oz/O2 indices for the EEG manuscript examples.
PATCH_SAMPLES=25
SEED=2023


In [ ]:
measured=responses(DATA)
encoder,decoder=load_model(CHECKPOINT,MODALITY,DEVICE)
prototypes=np.load(PROTOTYPES,allow_pickle=False)
assert prototypes.shape==(len(measured),1024)
def make_mask(shape,kind,ratio,seed):
    rng=np.random.default_rng(seed)
    n,c,t=shape; missing=np.zeros(shape,dtype=bool)
    for i in range(n):
        if kind=='spatial':
            missing[i,rng.choice(c,int(np.ceil(ratio*c)),replace=False),:]=True
        elif kind=='forecasting':
            missing[i,:,t-int(np.ceil(ratio*t)):]=True
        else:
            starts=list(range(0,t,PATCH_SAMPLES))
            count=len(starts) if kind=='temporal' else c*len(starts)
            chosen=rng.choice(count,int(np.ceil(ratio*count)),replace=False)
            for p in chosen:
                ch=p//len(starts); start=starts[p%len(starts)]
                if kind=='temporal': missing[i,:,start:start+PATCH_SAMPLES]=True
                else: missing[i,ch,start:start+PATCH_SAMPLES]=True
    return missing
records=[]; examples={}
for kind in KINDS:
    for ratio in RATIOS:
        missing=make_mask(measured.shape,kind,ratio,SEED)
        corrupted=np.where(missing,0.,measured)
        z=infer(encoder,corrupted[:,None],DEVICE)
        recon=infer(decoder,z,DEVICE)[:,0]
        # Match manuscript wording: classification after re-encoding reconstructed signals.
        reconstructed_z=infer(encoder,recon[:,None],DEVICE)
        record={'mask':kind,'nominal_ratio':ratio,'actual_ratio':missing.mean(),
                'missing_mse':np.mean((measured[missing]-recon[missing])**2),
                'observed_mse':np.mean((measured[~missing]-recon[~missing])**2),
                'missing_r':np.nanmean(pearson_rows(measured,recon,missing)),
                'full_r':np.nanmean(pearson_rows(measured,recon)),
                'classification_reconstructed':accuracy(reconstructed_z,prototypes,PROTOTYPE_METRIC),
                'classification_corrupted':accuracy(z,prototypes,PROTOTYPE_METRIC)}
        records.append(record)
        if np.isclose(ratio,.5): examples[kind]=(corrupted,recon,missing)
results=pd.DataFrame(records)
OUTPUT.mkdir(parents=True,exist_ok=True)
results.to_csv(OUTPUT/'reconstruction_masking.csv',index=False)
print(CHECKPOINT_TRAINING)
results


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(9,2.7),layout='constrained')
for ax,metric,label in zip(axes,['missing_mse','missing_r','classification_reconstructed'],
    ['Missing-entry MSE','Missing-entry Pearson r','Reconstructed-signal Top-1 (%)']):
    for j,kind in enumerate(KINDS):
        part=results[results['mask']==kind].sort_values('nominal_ratio')
        scale=100 if metric.startswith('classification') else 1
        ax.plot(100*part.actual_ratio,scale*part[metric],marker=['o','s','^','D'][j],
                color=[BLUE,RED,GRAY,'#4b8775'][j],label=kind)
    ax.set(xlabel='Masked entries (%)',ylabel=label)
axes[0].legend(fontsize=7)
save(fig,OUTPUT,'reconstruction_masking')
if examples:
    fig,axes=plt.subplots(len(EXAMPLE_CHANNELS),len(examples),figsize=(10,2*len(EXAMPLE_CHANNELS)),squeeze=False,layout='constrained')
    times=T0+np.arange(measured.shape[-1])/SFREQ
    for col,(kind,(corrupted,recon,missing)) in enumerate(examples.items()):
        for row,ch in enumerate(EXAMPLE_CHANNELS):
            ax=axes[row,col]; i=EXAMPLE_IMAGE
            ax.plot(times,measured[i,ch],color=BLUE,label='Measured')
            ax.plot(times,corrupted[i,ch],color=GRAY,alpha=.6,label='Masked input')
            ax.plot(times,recon[i,ch],color=RED,ls='--',label='Reconstructed')
            ax.fill_between(times,0,1,where=missing[i,ch],transform=ax.get_xaxis_transform(),color=GRAY,alpha=.12)
            ax.set(title=f'{kind}; channel {ch}',xlabel='Time (s)',ylabel='Amplitude (preprocessed)')
    axes[0,0].legend(fontsize=7)
    save(fig,OUTPUT,'reconstruction_waveforms')
